In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""
os.environ["GOOGLE_API_KEY"] = ""

#### Install Libraries

In [28]:
!pip install -q \
    youtube-transcript-api \
    langchain-community \
    langchain_google_genai \
    langchain-huggingface \
    faiss-cpu \
    tiktoken \
    python-dotenv

In [29]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

### 1. Indexing

In [54]:
video_id = "xAt1xcC6qfM"
try:
    ytt_api = YouTubeTranscriptApi()
    fetched_transcript = ytt_api.fetch(video_id, languages=["en"])
    transcript = " ".join(chunk.text for chunk in fetched_transcript)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

what's the biggest misunderstanding about you whenever you hear about somebody who's got you know ridiculous amounts of money their values are different than your values you should be concerned about their agenda what's your biggest Fe I'll be sad as my brain gets less capable that disappoints me tell me one Behavior we all should adopt reading a lot being a student that's a big part of My Success richest and most powerful men in the world Bill Gates has Unleashed a technological Revolution that has changed our lives Bill Gates of 25 and Bill Gates of 17 any change that you feel personally in my 20s being a maniac was the right thing my competitors would say oh no you work too hard and I'd say yes I do if you're in a race your 20s when you have no wife and no children that's the time to do it if you get an opportunity to invite three Indians for dinner who would that be there was a mathematician ramanujan I would have loved to have met him why do you think India is becoming a Global Ta

#### 1.1 Text-Splitter

In [31]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.create_documents([transcript])

In [32]:
len(chunks)

42

42

#### 1.2 Embedding

In [55]:
embeddings = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')
vector_stores = FAISS.from_documents(chunks, embeddings)

In [56]:
vector_stores.index_to_docstore_id

{0: '31cf1efa-a299-4f2e-ba20-e95d172fd0b3',
 1: '0e9333df-f54f-4a58-9c1f-58665559c401',
 2: '8f1d66e1-5dea-4cd5-8b28-ce9e42bd8554',
 3: '68eeda02-9eab-4f44-86e9-cc1ef5bde168',
 4: '3a90c520-c5a9-44d9-8d33-1f53f87c089e',
 5: '668bb251-284c-4726-b486-12c4bdc5312c',
 6: '155028d9-e125-4ae2-b6c8-5a51a6b9131b',
 7: 'bd9e5b8e-5244-4ce7-9781-78863a502bc0',
 8: 'c0ef384c-edf2-46d7-98b0-50df0fbc89a3',
 9: '6fd97521-d675-4353-b7de-55ac2fd72c48',
 10: '4fdc1fd9-03de-4e34-9cb4-dc9684a1c4a9',
 11: '11359d13-29ed-42be-b59a-5fe629d11b4f',
 12: '82542391-36b1-46ae-90fc-3ef84ed63b3a',
 13: '049a4b69-5d14-4d7d-aa99-b50dc285b4f4',
 14: '98ec13ca-55d7-4f18-a6ad-be619ba1826c',
 15: '97143a5e-9930-47bc-8f4b-4150e5c40f56',
 16: 'ee4e05a1-9fdd-4455-8130-35b870dfdb8e',
 17: '226c9c8f-1996-46f5-82fc-de71f4ff3ed4',
 18: '35fdf449-9e45-4555-b5ff-09a3a1b64230',
 19: 'cbf3e2d9-f084-41e9-9296-b6b8aa2c765b',
 20: '404f9ca6-3fea-4539-8f99-4466c096ee3c',
 21: 'cbc52e8b-06ff-4333-998f-ddbfc2f0255a',
 22: 'c3c55050-2c81-

In [57]:
vector_stores.get_by_ids(['c3c55050-2c81-49c8-84ab-370835f8555d'])

[Document(id='c3c55050-2c81-49c8-84ab-370835f8555d', metadata={}, page_content="you know not many countries have gotten the behavior change you know maybe India can Pioneer some approaches there but you know frankly and I'll sound like a technologist the most promising thing is actually uh a you know a drug a class of drugs called these glp1 drugs that you know are going to go off patent and become cheap and you know so I always like a you know I'm a little overfocused on a a scientific solution so maybe a combination of that behavior change and and the new tools but Behavior change is hard um we we haven't uh succeeded in that as as much as we'd like to that me one Behavior we all should adopt well you know the behavior that's helped me is is basically being a student all the time wanting to learn things and being pretty brutal with myself of do I really understand you know what's going on you know do I understand some AI thing or uh some disease thing and you know fortunately uh I ca

### 2. Retriver

In [58]:
retriever = vector_stores.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

In [59]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x795d038de240>, search_kwargs={'k': 4})

In [60]:
retriever.invoke('What are the mistake that is made by Bill Gates ?')

[Document(id='ee4e05a1-9fdd-4455-8130-35b870dfdb8e', metadata={}, page_content="out of 10,000 and you know would I be able to do that again hard to say you you talked about like when you started a lot of people thought that what are you doing this is some random thing these guys are trying to do they may be stupid they may be thinking too big whatever right and over the time every time you do something big usually people do not understand you or anyone like not just you like whoever wants to start something new and big in today's world or as of today what's the biggest misunderstanding about you what do you think what do people misunderstand about Bill Gates well whenever you hear about somebody who's got you know some degree of power or you know ridiculous amounts of money you know might you might think they have grand schemes uh you know and they're uh you know there's almost a sense that you know their their values are different than your values and you know that you should be conce

### Augumentation

In [61]:
prompt = PromptTemplate(
    template="""
      You are a Helful assistant.
      Use the following pieces of context to answer the question at the end.
      If the context is insufficents, just says I don't Know

      {context}
      Question: {question}
      """,

    input_variables=["context", "question"]
)

In [62]:
question = "is the topic discussed in this video ? if yes , then what was discussed give me the valuable points for that topic only"
retriever_docs= retriever.invoke(question)

In [67]:
context_text = "\n\n".join(docs.page_content for docs in retriever_docs)
context_text

"about this man about the kind of money he has made the kind of lives he's impacting the kind of things he's building through Microsoft from there to be sitting in front of him it was surreal when I started the conversation you could see it on my face that I was really nervous really scared I didn't know what to talk about but as we went in the conversation the podcaster in me took over and we spoke about his fears his Mis understandings the mission and what is he doing today today I want you to see this episode from a lens of a 20-year-old sitting with Bill Gates and figuring out what goes on in his brain this episode is truly special because I could have never thought that Bill Gates will be on our podcast this soon in our journey I always had a belief that we will be able to sit down with the smartest people around the world but it happen this soon I can't believe this right now I just want you to enjoy this episode the way I did I want you to sink in that this is happening because\

In [68]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [69]:
final_prompt

StringPromptValue(text="\n      You are a Helful assistant.\n      Use the following pieces of context to answer the question at the end.\n      If the context is insufficents, just says I don't Know\n\n      about this man about the kind of money he has made the kind of lives he's impacting the kind of things he's building through Microsoft from there to be sitting in front of him it was surreal when I started the conversation you could see it on my face that I was really nervous really scared I didn't know what to talk about but as we went in the conversation the podcaster in me took over and we spoke about his fears his Mis understandings the mission and what is he doing today today I want you to see this episode from a lens of a 20-year-old sitting with Bill Gates and figuring out what goes on in his brain this episode is truly special because I could have never thought that Bill Gates will be on our podcast this soon in our journey I always had a belief that we will be able to sit

### Generation

In [70]:
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.7,
)

In [71]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic discussed is Bill Gates' work and concerns.  Valuable points include:

* **Challenges in philanthropic work:**  Gates discusses the setbacks faced by the Bill & Melinda Gates Foundation in areas like HIV vaccine development, affordable sanitation ("cheap toilet"), and polio eradication.  He highlights that these projects are taking longer and proving more challenging than anticipated.

* **Ongoing challenges in global health:**  He points to malnutrition as a key area where understanding and solutions are still lacking, emphasizing the need for better seeds and agricultural advice.  He also mentions the importance of continued implementation of solutions, particularly in India and Africa.

* **South-South collaboration:** Gates highlights the foundation's role in facilitating collaboration between developing nations (e.g., India and African countries) in areas like digital public infrastructure, sharing knowledge and resources.

* **Predicting and preventing pandemics:**

# Building a Chain

In [72]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [73]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [74]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs), # context string
    'question': RunnablePassthrough()
})

In [76]:
parallel_chain.invoke('Who is Bill Gate & What was he built ?')

{'context': "about this man about the kind of money he has made the kind of lives he's impacting the kind of things he's building through Microsoft from there to be sitting in front of him it was surreal when I started the conversation you could see it on my face that I was really nervous really scared I didn't know what to talk about but as we went in the conversation the podcaster in me took over and we spoke about his fears his Mis understandings the mission and what is he doing today today I want you to see this episode from a lens of a 20-year-old sitting with Bill Gates and figuring out what goes on in his brain this episode is truly special because I could have never thought that Bill Gates will be on our podcast this soon in our journey I always had a belief that we will be able to sit down with the smartest people around the world but it happen this soon I can't believe this right now I just want you to enjoy this episode the way I did I want you to sink in that this is happen

In [77]:
parser = StrOutputParser()

In [78]:
main_chain = parallel_chain | prompt | llm | parser

In [79]:
main_chain.invoke('Can you summarize the video')

"This video is a podcast interview with Bill Gates.  The host, visibly nervous and excited at the beginning, expresses disbelief at having secured such a high-profile guest.  He recounts a prediction he made during the pandemic that he would one day interview Bill Gates.  The interview itself covers Gates's fears, misunderstandings, mission, and current work. The host emphasizes the special nature of the interview and encourages viewers to subscribe and suggest future guests.  He also mentions Bill Gates's impact and the respect he commanded even during the host's childhood."